# Part D — Linear Transformations

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA**

Preprocessing steps like standardization aren't just "cleaning" — they are linear maps with their own matrix representation.

---
**Student name:** Alfajaro, Ghan Vincent | Arile, Charles Jullianne | Asuar, Jhoe Maven | Mondragon, Leneth | Ulitin, Justin | **Date:** November 16, 2026

---


**What to do:** Define at least one linear transformation (e.g., standardization, normalization, or a projection) and apply it to your data.

In [1]:
# Run this first: recall variables saved from the previous part's notebook.
# (If this is your first notebook in the session, or the store is empty,
#  it fetches directly from the official BSP URL so no CSV upload is needed.)
%store -r X y feature_names rank_X

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [3]:
# Confirm Part C's full column rank result before constructing the standardization map
assert rank_X == X.shape[1], f"Expected full column rank {X.shape[1]}, got {rank_X}."
print(f"Verified from Part C: rank(X) = {rank_X} = p (full column rank, no redundant features).\n")

# Chronological 80/20 split boundary so mu and sigma are computed ONLY from
# the training set (prevents future-data leakage per the project proposal).
train_size = int(0.8 * X.shape[0])
X_train_raw = X[:train_size]

# Build the matrix representation of standardization: T(x) = S(x - mu)
mu = X_train_raw.mean(axis=0)
sigma = X_train_raw.std(axis=0, ddof=0)
S = np.diag(1.0 / sigma)   # 3x3 diagonal scaling matrix S = diag(1/sigma)

X_centered = X - mu
X_transformed = X_centered @ S   # Apply T(x) = S(x - mu) across all rows

print("Training means (mu):", mu)
print("Training stds (sigma):", sigma)
print("\nScaling transformation matrix S = diag(1/sigma):")
print(S)
print("\nX_transformed shape:", X_transformed.shape)

Verified from Part C: rank(X) = 3 = p (full column rank, no redundant features).

Training means (mu): [42.4302 42.3602  0.9993]
Training stds (sigma): [9.942  9.9434 3.864 ]

Scaling transformation matrix S = diag(1/sigma):
[[0.1006 0.     0.    ]
 [0.     0.1006 0.    ]
 [0.     0.     0.2588]]

X_transformed shape: (428, 3)


In [4]:
# Show the transformation applied to ONE real data point (index 0 = Jan 1991)
idx = 0
x_before = X[idx]
x_after = X_transformed[idx]

print(f"Sample index {idx} (1991-01):")
print("Before (x):              ", x_before)
print("Centered (x - mu):       ", x_before - mu)
print("After T(x) = S(x - mu):  ", x_after)

# Verify invertibility: x = S^-1 * T(x) + mu
S_inv = np.diag(sigma)
x_recovered = (S_inv @ x_after) + mu
print("Recovered (S^-1 z + mu): ", x_recovered)
print("Exact inverse recovery?  ", np.allclose(x_before, x_recovered))

Sample index 0 (1991-01):
Before (x):               [28.     28.      5.5356]
Centered (x - mu):        [-14.4302 -14.3602   4.5363]
After T(x) = S(x - mu):   [-1.4514 -1.4442  1.174 ]
Recovered (S^-1 z + mu):  [28.     28.      5.5356]
Exact inverse recovery?   True


In [5]:
# Chronological 80/20 train/test split (no random shuffle, preserving time order)
n = X_transformed.shape[0]
train_size = int(0.8 * n)

X_train, X_test = X_transformed[:train_size], X_transformed[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("X_train shape:", X_train.shape, "| y_train shape:", y_train.shape)
print("X_test shape: ", X_test.shape,  "| y_test shape: ", y_test.shape)
print("X_train column means (should be ~0):", X_train.mean(axis=0))
print("X_train column stds  (should be ~1):", X_train.std(axis=0))

X_train shape: (342, 3) | y_train shape: (342,)
X_test shape:  (86, 3) | y_test shape:  (86,)
X_train column means (should be ~0): [-0. -0.  0.]
X_train column stds  (should be ~1): [1. 1. 1.]


### Answer these in your report

**Q13.** Which linear transformation did you apply to your data? Write its matrix representation M such that T(x) = Mx.

> *Your answer:* We applied feature standardization (z-score scaling) using the training set's feature means (μ = [42.4302, 42.3602, 0.9993]) and standard deviations (σ = [9.9420, 9.9434, 3.8640]). Because subtracting the mean vector μ shifts the origin, full standardization is technically an affine map T_affine(x) = S(x - μ). Once the observations are centered as x̃ = x - μ, the purely linear transformation is T(x̃) = Mx̃, where M = S is a 3 × 3 diagonal scaling matrix whose diagonal entries are the reciprocals of the training standard deviations (1/σ_j):
>
> $$M = S = \begin{bmatrix} 1/\sigma_1 & 0 & 0 \\ 0 & 1/\sigma_2 & 0 \\ 0 & 0 & 1/\sigma_3 \end{bmatrix} = \begin{bmatrix} 0.1006 & 0 & 0 \\ 0 & 0.1006 & 0 \\ 0 & 0 & 0.2588 \end{bmatrix}$$
>
> Multiplying by M rescales each feature so that all three columns have unit variance (σ = 1) in the training set. This puts the PHP/USD rate levels (`CurrentRate` and `Trailing3MonthAverage`) and the difference feature (`YearOverYearChange`) on the same dimensionless scale before covariance and PCA calculations.

**Q14.** Using one actual data point from your dataset, show the before vector x and the after vector T(x) = Mx.

> *Your answer:* Using the first observation in our dataset (`idx = 0`, representing January 1991, which predicts February 1991), the raw feature vector before transformation is:
>
> $$x = \begin{bmatrix} 28.0000 \\ 28.0000 \\ 5.5356 \end{bmatrix}$$
>
> After subtracting the training set mean vector μ to get the centered vector x̃ = x - μ = [-14.4302, -14.3602, 4.5363]ᵀ, and multiplying by our scaling matrix M = S, we obtain the standardized vector:
>
> $$T(x) = S(x - \mu) = \begin{bmatrix} -1.4514 \\ -1.4442 \\ 1.1740 \end{bmatrix}$$
>
> In words, each entry in T(x) tells us how many training-set standard deviations away from the historical training mean January 1991 sat for that feature. Because 1991 exchange rates were around ₱28/USD—well below the multi-decade training average—X1 and X2 become negative z-scores (-1.4514 and -1.4442), whereas the sharp peso depreciation between January 1990 and January 1991 (+5.5356) gives X3 a positive z-score (+1.1740).

**Q15.** Is your transformation invertible? How do you know, and why would invertibility matter if you wanted to convert a prediction back to original units?

> *Your answer:* Yes, our transformation is invertible. Because M = S is a diagonal matrix, its determinant is simply the product of its diagonal entries: det(S) = 1 / (σ_1 · σ_2 · σ_3). Since all three features vary across months (consistent with Part C's full column rank of 3), every training standard deviation σ_j is strictly greater than zero, meaning det(S) ≠ 0 and S has full rank (3). Its inverse is simply S⁻¹ = diag(σ_1, σ_2, σ_3), so any standardized vector z = S(x - μ) can be mapped back exactly to original PHP/USD units via x = S⁻¹z + μ.
>
> Invertibility matters whenever we need to interpret model outputs or coefficients in real-world currency units. If a model operates on or predicts standardized values, or if we want to translate regression weights back into how a ₱1.00 change in the current rate affects next month's rate, an invertible transformation guarantees a one-to-one mapping with zero information loss.

**Q16.** Would your transformation's matrix change if computed from a different subset of your data (e.g., a different train/test split)? What does that imply about how you must apply it consistently to both training and test data?

> *Your answer:* Yes, M = S would change if computed from a different subset of the data because the sample mean μ and sample standard deviation σ depend directly on which months are included in the subset. For example, the peso traded around ₱24–₱28/USD in the early 1990s but rose above ₱55–₱61/USD in the 2020s, so changing the chronological train/test cutoff shifts both the mean and the spread of the training window.
>
> This implies two strict rules for applying the transformation:
> 1. **Fit only on the training set:** We must compute μ and σ exclusively from `X_train` and never from `X_test` or the combined dataset. Computing them on the full dataset would leak future exchange-rate levels and volatility from the test period into the training process.
> 2. **Apply the exact same parameters to the test set:** Once S and μ are fixed from `X_train`, we must use those exact same training constants to transform `X_test`. If we standardized `X_test` using its own mean and standard deviation, a standardized value of 0 in the test set would represent a completely different peso rate than 0 in the training set, invalidating our regression predictions.

### Save your work for the next notebook

In [6]:
%store X_transformed
%store X_train
%store X_test
%store y_train
%store y_test
%store mu
%store sigma
%store S
print("Saved X_transformed, X_train, X_test, y_train, y_test (plus mu, sigma, S) for Part E.")

Stored 'X_transformed' (ndarray)
Stored 'X_train' (ndarray)
Stored 'X_test' (ndarray)
Stored 'y_train' (ndarray)
Stored 'y_test' (ndarray)
Stored 'mu' (ndarray)
Stored 'sigma' (ndarray)
Stored 'S' (ndarray)
Saved X_transformed, X_train, X_test, y_train, y_test (plus mu, sigma, S) for Part E.
